In [0]:
from pyspark.sql.functions import *

raw_df = spark.range(0, 100).withColumn("order_id", col("id") + 5000) \
    .withColumn("product", element_at(array(lit("Phone"), lit("Tablet"), lit("Laptop")), (rand()*3 + 1).cast("int"))) \
    .withColumn("amount", (rand() * 1000).cast("decimal(10,2)")) \
    .withColumn("ingestion_timestamp", current_timestamp())

raw_df.write.format("delta").mode("overwrite").saveAsTable("orders_bronze")

display(spark.table("orders_bronze"))

In [0]:
from pyspark.sql.functions import upper, col

bronze_df = spark.table("orders_bronze")

silver_df = bronze_df.select(
    "order_id",
    upper(col("product")).alias("product_name"),
    col("amount"),
    col("ingestion_timestamp")
).filter(col("amount") > 500)

silver_df.write.format("delta").mode("overwrite").saveAsTable("orders_silver")

print("✨ Silver Layer is Ready! High-value orders filtered.")
display(spark.table("orders_silver"))

In [0]:

silver_table = spark.table("orders_silver")

gold_df = silver_table.groupBy("product_name").agg(
    sum("amount").alias("total_revenue"),
    count("order_id").alias("number_of_orders"),
    round(avg("amount"), 2).alias("avg_order_value")
).orderBy(col("total_revenue").desc())

gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("sales_summary_gold")

print("🏆 GOLD LAYER COMPLETE: Business Insights Generated!")
display(spark.table("sales_summary_gold"))

# Final Comment

In [0]:
raw_data_df = spark.table("workspace.default.raw_sales_data")

from pyspark.sql.functions import current_timestamp

bronze_df = raw_data_df.withColumn("ingestion_timestamp", current_timestamp())

bronze_df.write.format("delta").mode("overwrite").saveAsTable("sales_raw_bronze")

print("Bronze Layer: Success! Raw data ingested into Delta Lake.")
display(spark.table("sales_raw_bronze"))

In [0]:
from pyspark.sql.functions import col, upper, expr

bronze_data = spark.table("sales_raw_bronze")

silver_df = bronze_data \
    .withColumn("amount", expr("try_cast(amount as double)")) \
    .filter(col("amount").isNotNull()) \
    .withColumn("customer_name", upper(col("customer_name"))) \
    .filter(col("amount") > 500)

silver_df.write.format("delta").mode("overwrite").saveAsTable("sales_cleaned_silver")

print("🥈 Silver Layer: FIXED! Using try_cast to handle malformed strings.")
display(spark.table("sales_cleaned_silver"))


In [0]:
from pyspark.sql.functions import sum, avg, round

silver_data = spark.table("sales_cleaned_silver")

gold_df = silver_data \
    .groupBy("product") \
    .agg(sum("amount").alias("total_revenue"), avg("amount").alias("avg_amount")) \
    .withColumn("total_revenue", round(col("total_revenue"), 2)) \
    .withColumn("avg_amount", round(col("avg_amount"), 2))

gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("sales_summary_gold")

print("🥇 GOLD LAYER COMPLETE: Business Insights Generated!")
display(spark.table("sales_summary_gold"))

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

customers_data = [
    (1, "Aung Aung", "Yangon", "2023-01-15"),
    (2, "Su Su", "Mandalay", "2023-02-20"),
    (3, "Kyaw Kyaw", "Yangon", "2023-03-10"),
    (4, "Hla Hla", "Taunggyi", "2023-04-05")
]

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("join_date", StringType(), True)
])

customers_df = spark.createDataFrame(customers_data, schema)
customers_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.customers")

print("✅ Customer data saved successfully to: workspace.default.customers")
display(spark.table("workspace.default.customers"))

In [0]:
print(f"Silver Sales Count: {spark.table('sales_cleaned_silver').count()}")
print(f"Customers Count: {spark.table('workspace.default.customers').count()}")

test_join = spark.table("sales_cleaned_silver").join(spark.table("workspace.default.customers"), "customer_name")
print(f"After Join Count: {test_join.count()}")

In [0]:
from pyspark.sql.functions import lower, trim, col, broadcast, sum, rank, desc
from pyspark.sql.window import Window

silver_cleaned = spark.table("sales_cleaned_silver") \
    .withColumn("join_key", lower(trim(col("customer_name"))))

customers_cleaned = spark.table("workspace.default.customers") \
    .withColumn("join_key", lower(trim(col("customer_name"))))

sales_with_customer = silver_cleaned.join(broadcast(customers_cleaned), "join_key")

window_spec = Window.partitionBy("city").orderBy(desc("total_revenue"))

city_top_products = sales_with_customer.groupBy("city", "product") \
    .agg(sum("amount").alias("total_revenue")) \
    .withColumn("rank", rank().over(window_spec)) \
    .filter(col("rank") == 1)

city_top_products.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.city_top_products_gold")

print(f"✅ SUCCESS: Join count now is {sales_with_customer.count()}")
display(spark.table("workspace.default.city_top_products_gold"))